# 18. 策略梯度与 Actor-Critic 体系
当动作空间较大、策略需要显式随机性、或值函数方法难以稳定做 $\arg\max$ 时，直接在策略空间中优化就成为更自然的选择。策略梯度方法不再先学“动作值表”，而是直接沿着“让高回报动作概率变大”的方向更新参数。

这条路线从 REINFORCE 出发，引入 baseline 降方差，进一步发展为 Actor-Critic、A2C/A3C、TRPO 与 PPO，构成现代策略优化的主干。

## 策略梯度与 Actor-Critic的形式化定义

            策略梯度方法直接参数化策略 $\pi_\theta(a\mid s)$，目标是最大化
$$
J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\left[\sum_{t=0}^{T}\gamma^t r_t\right].
$$
策略梯度定理给出
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\pi_\theta}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t) Q^{\pi_\theta}(s_t,a_t)\right].
$$
它说明策略更新并不需要对环境转移概率求导，只需要对策略的对数概率求导并乘上回报或优势信号。

## 输入、输出与参数化方式

            输入是状态 $s$，输出是动作分布参数：

- 离散动作场景中，输出通常是 logits，再通过 softmax 得到 $\pi_\theta(a\mid s)$；
- 连续动作场景中，输出通常是高斯分布的均值与方差。

如果再加入价值函数估计器，就得到 Actor-Critic 结构：Actor 输出策略，Critic 输出 $V(s)$ 或 $Q(s,a)$，用于降低梯度估计方差并提供更稳定的信用分配信号。

## 结构分解与信息流

            REINFORCE 的结构最简单：采样一整条轨迹，计算每一步的回报 $G_t$，然后让提高回报的动作概率上升。它的优点是无偏，缺点是方差大、学习慢。

Actor-Critic 的关键思想是把“谁负责决策”和“谁负责评价”拆开：

- Actor：输出动作分布并执行动作；
- Critic：估计状态价值或动作价值；
- Advantage / TD error：把 Critic 的估计转成 Actor 的学习信号。

TRPO 和 PPO 则进一步在策略更新时显式限制新旧策略之间的偏移，避免一步更新过大导致性能崩塌。

## 优化目标与训练机制

            常见目标函数包括：

- REINFORCE：
  $$
  \mathcal{L}_{\text{pg}} = - \mathbb{E}\left[\log \pi_\theta(a_t\mid s_t) G_t\right]
  $$
- 加 baseline 后：
  $$
  \mathcal{L}_{\text{pg}} = - \mathbb{E}\left[\log \pi_\theta(a_t\mid s_t)\left(G_t - b(s_t)\right)\right]
  $$
- Actor-Critic 常用 TD 优势：
  $$
  \hat{A}_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)
  $$
- PPO 的 clipped surrogate：
  $$
  \mathcal{L}_{\text{PPO}} =
  \mathbb{E}\left[\min\left(r_t(\theta)\hat{A}_t, \operatorname{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right]
  $$
  其中 $r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\text{old}}}(a_t\mid s_t)}$。

## 计算复杂度、统计性质与工程代价

            策略梯度方法的主要代价来自高方差与 on-policy 采样需求。相比离策略值函数方法，它们通常样本效率更低，但在处理随机策略、连续动作和受约束更新时更加自然。

Actor-Critic 的稳定性高度依赖优势估计、价值函数偏差、熵正则、学习率和 mini-batch 重复使用策略。PPO 之所以广泛使用，并不是因为它在理论上最强，而是因为它在“稳定性 - 实现复杂度 - 性能”之间取得了较好的折中。

## 与相邻模型的差异

REINFORCE 是最基础的无偏策略梯度；Actor-Critic 通过 Critic 降低方差；A2C/A3C 主要是并行化与同步/异步训练框架；TRPO 用近似信赖域保证更新保守；PPO 用 clipping 近似 TRPO 的约束思想，以更简单的实现获得较好的稳定性，因此成为工业与研究中最常见的策略优化基线。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5.5)

blocks = [
    (1.4, 2.75, 1.6, 1.0, "#4C78A8", "state s"),
    (4.2, 3.7, 2.2, 1.0, "#F58518", "actor\npi(a|s)"),
    (4.2, 1.8, 2.2, 1.0, "#54A24B", "critic\nV(s) / Q(s,a)"),
    (7.2, 2.75, 2.0, 1.0, "#E45756", "action a"),
    (10.2, 2.75, 2.3, 1.2, "#72B7B2", "advantage / TD error"),
]
for x, y, w, h, color, text in blocks:
    rect = plt.Rectangle((x - w / 2, y - h / 2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(3.1, 3.7), xytext=(2.2, 2.95), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(3.1, 1.8), xytext=(2.2, 2.55), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(6.2, 2.75), xytext=(5.3, 3.35), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(9.0, 2.75), xytext=(8.2, 2.75), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(5.3, 2.15), xytext=(9.1, 2.25), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.text(8.2, 4.45, "critic supplies the learning signal for the actor", ha="center", fontsize=11)
ax.set_title("Actor-Critic Information Flow")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

ax.add_patch(plt.Rectangle((0.8, 1.8), 2.2, 1.4, facecolor="#4C78A8", edgecolor="black"))
ax.text(1.9, 2.5, "old policy rollout\npi_old", ha="center", va="center", fontsize=11)
ax.add_patch(plt.Rectangle((4.0, 1.8), 2.0, 1.4, facecolor="#F58518", edgecolor="black"))
ax.text(5.0, 2.5, "ratio r = pi / pi_old", ha="center", va="center", fontsize=11)
ax.add_patch(plt.Rectangle((7.0, 1.8), 2.2, 1.4, facecolor="#E45756", edgecolor="black"))
ax.text(8.1, 2.5, "clip to [1-eps, 1+eps]", ha="center", va="center", fontsize=11)
ax.add_patch(plt.Rectangle((10.2, 1.8), 2.0, 1.4, facecolor="#54A24B", edgecolor="black"))
ax.text(11.2, 2.5, "stable update", ha="center", va="center", fontsize=11)

for start, end in [(3.0, 4.0), (6.0, 7.0), (9.2, 10.2)]:
    ax.annotate("", xy=(end, 2.5), xytext=(start, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))

ax.text(6.0, 4.0, "PPO keeps policy improvement but limits destructive update steps", ha="center", fontsize=12)
ax.set_title("Clipped PPO Objective")
plt.show()

## 优势函数、GAE 与稳定更新

在现代策略优化中，优势函数是连接 Critic 与 Actor 的关键桥梁。它回答的问题不是“这个动作绝对值多少钱”，而是“这个动作比当前状态的平均动作好多少”。如果只用回报 $G_t$ 训练策略，方差很大；如果用价值函数差分构造优势，方差会显著降低。

广义优势估计（GAE）进一步把多步 TD 误差按 $\lambda$ 衰减累加：
$$
\hat{A}^{\text{GAE}(\gamma,\lambda)}_t = \sum_{l=0}^{\infty}(\gamma\lambda)^l \delta_{t+l},
\qquad
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t).
$$
其中 $\lambda$ 控制偏差与方差之间的折中：$\lambda$ 越大，越接近 Monte Carlo；$\lambda$ 越小，越接近一步 TD。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
class CorridorEnv:
    def __init__(self, n_states=7, max_steps=15):
        self.n_states = n_states
        self.max_steps = max_steps
        self.reset()

    def _obs(self):
        obs = np.zeros(self.n_states, dtype=np.float32)
        obs[self.pos] = 1.0
        return obs

    def reset(self):
        self.pos = 0
        self.steps = 0
        return self._obs()

    def step(self, action):
        self.steps += 1
        self.pos = int(np.clip(self.pos + (1 if action == 1 else -1), 0, self.n_states - 1))
        reward = 1.0 if self.pos == self.n_states - 1 else -0.02
        done = self.pos == self.n_states - 1 or self.steps >= self.max_steps
        return self._obs(), reward, done

class PolicyNet(nn.Module):
    def __init__(self, input_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.Tanh(),
            nn.Linear(32, action_dim),
        )

    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_reinforce(episodes=260, gamma=0.98):
    env = CorridorEnv()
    policy = PolicyNet(env.n_states, 2)
    optimizer = torch.optim.Adam(policy.parameters(), lr=2e-3)
    returns = []

    for _ in range(episodes):
        state = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits = policy(state_tensor)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            next_state, reward, done = env.step(int(action.item()))
            log_probs.append(dist.log_prob(action))
            rewards.append(reward)
            state = next_state

        discounted_returns = []
        G = 0.0
        for reward in reversed(rewards):
            G = reward + gamma * G
            discounted_returns.insert(0, G)
        returns_tensor = torch.tensor(discounted_returns, dtype=torch.float32)
        returns_tensor = (returns_tensor - returns_tensor.mean()) / (returns_tensor.std() + 1e-6)

        loss = 0.0
        for log_prob, ret in zip(log_probs, returns_tensor):
            loss = loss - log_prob * ret

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        returns.append(sum(rewards))
    return np.array(returns)

def train_actor_critic(episodes=260, gamma=0.98):
    env = CorridorEnv()
    policy = PolicyNet(env.n_states, 2)
    value = ValueNet(env.n_states)
    policy_opt = torch.optim.Adam(policy.parameters(), lr=2e-3)
    value_opt = torch.optim.Adam(value.parameters(), lr=2e-3)
    returns = []

    for _ in range(episodes):
        state = env.reset()
        episode_return = 0.0
        done = False
        while not done:
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits = policy(state_tensor)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            next_state, reward, done = env.step(int(action.item()))
            next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0)

            value_s = value(state_tensor)
            with torch.no_grad():
                value_next = value(next_state_tensor) if not done else torch.tensor([0.0])
                td_target = torch.tensor([reward], dtype=torch.float32) + gamma * value_next
                advantage = td_target - value_s.detach()

            actor_loss = -(dist.log_prob(action) * advantage).mean()
            critic_loss = nn.functional.mse_loss(value_s, td_target)

            policy_opt.zero_grad()
            actor_loss.backward()
            policy_opt.step()

            value_opt.zero_grad()
            critic_loss.backward()
            value_opt.step()

            state = next_state
            episode_return += reward
        returns.append(episode_return)
    return np.array(returns)

reinforce_returns = train_reinforce()
actor_critic_returns = train_actor_critic()
pg_df = pd.DataFrame(
    {
        "episode": np.arange(1, len(reinforce_returns) + 1),
        "REINFORCE": reinforce_returns,
        "Actor-Critic": actor_critic_returns,
    }
)
pg_df.head()

In [ ]:
pg_long = pg_df.melt(id_vars="episode", var_name="method", value_name="return")
pg_long["moving_avg"] = pg_long.groupby("method")["return"].transform(
    lambda x: pd.Series(x).rolling(15, min_periods=1).mean()
)
plt.figure(figsize=(10, 5))
sns.lineplot(data=pg_long, x="episode", y="moving_avg", hue="method")
plt.title("REINFORCE vs Actor-Critic")
plt.ylabel("moving average return")
plt.show()

In [ ]:
td_deltas = np.array([0.9, 0.4, -0.1, 0.7, -0.2])

def gae_from_deltas(deltas, gamma=0.99, lam=0.95):
    advantages = np.zeros_like(deltas, dtype=float)
    running = 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * running
        advantages[t] = running
    return advantages

gae_records = []
for lam in [0.0, 0.5, 0.95, 1.0]:
    advantages = gae_from_deltas(td_deltas, gamma=0.99, lam=lam)
    for step, advantage in enumerate(advantages):
        gae_records.append({"step": step, "lambda": lam, "advantage": advantage})
gae_df = pd.DataFrame(gae_records)

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=gae_df, x="step", y="advantage", hue="lambda", marker="o")
plt.title("Effect of lambda in Generalized Advantage Estimation")
plt.show()

ratios = np.linspace(0.0, 2.0, 200)
clip_eps = 0.2
ppo_df = pd.DataFrame({"ratio": ratios})
for advantage in [1.0, -1.0]:
    unclipped = ratios * advantage
    clipped = np.clip(ratios, 1 - clip_eps, 1 + clip_eps) * advantage
    ppo_df[f"adv_{advantage:+.0f}"] = np.minimum(unclipped, clipped)
ppo_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ppo_df["ratio"], ppo_df["adv_+1"], label="advantage > 0")
plt.plot(ppo_df["ratio"], ppo_df["adv_-1"], label="advantage < 0")
plt.axvline(1 - clip_eps, color="gray", linestyle="--")
plt.axvline(1 + clip_eps, color="gray", linestyle="--")
plt.title("Clipped PPO Surrogate as a Function of Policy Ratio")
plt.xlabel("probability ratio r")
plt.ylabel("surrogate objective term")
plt.legend()
plt.show()

## 方法比较

- **REINFORCE**：概念最纯粹，但方差最高，通常只作为入门与理论基线。
- **Actor-Critic**：通过 Critic 提供低方差学习信号，是现代 RL 的基础骨架。
- **TRPO**：强调近似信赖域约束，理论上更严格，但实现复杂。
- **PPO**：用 clipping 或 KL penalty 在近似保守更新的同时保持实现简单，因此成为最主流的在线策略优化基线之一。
- **A2C / A3C**：重点在并行采样和训练框架，而不是改变目标函数本身。

## 主要资料

- A3C（2016）: https://arxiv.org/abs/1602.01783
- TRPO（2015）: https://arxiv.org/abs/1502.05477
- PPO（2017）: https://arxiv.org/abs/1707.06347